# SalesOps Agent Prototype

This notebook is the starting prototype for the AgentOps project.

You are an AI Engineer at **UdaCenture**, a fictional B2B software company. The Sales Operations team has built a prototype agent that can inspect CRM data, reason about sales pipeline, summarize account risk, draft follow-up emails, and use internal tools.

The prototype is intentionally **not production-ready**. Your project is to operationalize it into a reproducible, versioned, evaluated, controlled, observable, and documented Python application.


## Dataset Model

The CRM uses **accounts**, not just customers.

An account can be a prospect, qualified lead, customer, or former customer. Only accounts with at least one `Closed Won` opportunity are considered customers in this starter dataset.

The data includes:

- 109 B2B accounts
- 27 customer accounts
- contacts for every account
- opportunities across prospects and customers
- activities for some accounts
- support tickets only for customers
- product usage only for customers
- restricted internal HR and financial data


> **⚠️ This notebook has serious operational problems.** Your mission is to identify them and transform this notebook into a robust, production-ready agentic system using LangGraph and industry best practices.

## Install note

This notebook assumes the workspace already has LangChain and an LLM provider configured. If your environment uses a different model, replace `openai:gpt-4o-mini` in the agent creation cell.


In [ ]:
from pathlib import Path
import json
from typing import Any, Dict, List, Optional
import re
from dotenv import load_dotenv
from langchain.agents import create_agent

In [ ]:
# Make sure you have your OpenAI API key set
# os.environ["OPENAI_API_KEY"] = "your-api-key"
# or inside you .env file
load_dotenv()

In [ ]:
def create_file_path(folder_name:str, file_name:str):
    return Path(folder_name) / file_name

In [ ]:
ACCOUNTS_PATH = create_file_path('data', 'accounts.json')
CONTACTS_PATH = create_file_path('data', 'contacts.json')
OPPORTUNITIES_PATH = create_file_path('data', 'opportunities.json')
ACTIVITIES_PATH = create_file_path('data', 'activities.json')
SUPPORT_TICKETS_PATH = create_file_path('data', 'support_tickets.json')
PRODUCT_USAGE_PATH = create_file_path('data', 'product_usage.json')
INTERNAL_HR_PATH = create_file_path('data', 'internal_hr_data.json')
INTERNAL_FINANCIAL_PATH = create_file_path('data', 'internal_financial_data.json')
EMAIL_OUTBOX_PATH = create_file_path('data', 'email_outbox.json')

In [ ]:
def load_json(path: Path) -> Any:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

In [ ]:
accounts = load_json(ACCOUNTS_PATH)

In [ ]:
contacts = load_json(CONTACTS_PATH)

In [ ]:
opportunities = load_json(OPPORTUNITIES_PATH)

In [ ]:
activities = load_json(ACTIVITIES_PATH)

In [ ]:
support_tickets = load_json(SUPPORT_TICKETS_PATH)

In [ ]:
product_usage = load_json(PRODUCT_USAGE_PATH)

In [ ]:
internal_hr_data = load_json(INTERNAL_HR_PATH)

In [ ]:
internal_financial_data = load_json(INTERNAL_FINANCIAL_PATH)

In [ ]:
print(f'Accounts: {len(accounts)}')
print(f'Customer accounts: {sum(1 for a in accounts if a["account_status"] == "customer")}')
print(f'Contacts: {len(contacts)}')
print(f'Opportunities: {len(opportunities)}')
print(f'Activities: {len(activities)}')
print(f'Support tickets: {len(support_tickets)}')
print(f'Product usage records: {len(product_usage)}')

In [ ]:
accounts[:3]

## Hardcoded System Prompt

This prompt is intentionally hardcoded in the notebook. In your final project, prompts should be versioned outside execution logic.


In [ ]:
SYSTEM_PROMPT = '''
You are a SalesOps Agent for UdaCenture, a B2B software company.

You help sales and revenue teams answer questions about:
- accounts,
- contacts,
- opportunities,
- customer risk,
- sales pipeline,
- recent activity,
- support issues,
- product usage,
- and customer follow-up emails.

Important domain definitions:
- An account is any B2B organization tracked by SalesOps.
- A customer is an account with at least one Closed Won opportunity.
- Not every account is a customer.
- Tickets and product usage should only exist for customer accounts.

When asked to summarize account risk, consider:
- account status,
- customer health score,
- contract end date,
- open renewal or expansion opportunities,
- opportunity stage,
- last activity,
- support tickets,
- product usage trend,
- relationship status of contacts,
- and activity notes.

When drafting emails:
- be professional,
- be concise,
- do not invent discounts, commitments, legal terms, or confidential facts,
- use the contact information available in the CRM data.

This is a prototype. You may use internal tools if needed.
'''

## Lookup Helper Functions

These helpers are intentionally simple. In the final project, you should move data access logic into versioned tools/modules.


In [ ]:
def normalize_lookup_text(value: str) -> str:
    """Normalize names and IDs for forgiving lookups in the prototype.

    This makes demo questions more robust to punctuation, casing, and common
    legal suffix variants such as "Inc." vs "Inc".
    """
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\b(incorporated|inc|corp|corporation|ltd|limited|llc|co|company)\b", "", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


In [ ]:
def find_account_by_id(account_id: str) -> Optional[Dict[str, Any]]:
    """Look up an account by exact account ID, case-insensitive."""
    account_id_normalized = account_id.strip().lower()
    return next((a for a in accounts if a['account_id'].lower() == account_id_normalized), None)

In [ ]:
def find_account_by_name(name: str) -> Optional[Dict[str, Any]]:
    """Look up an account by name using forgiving matching.

    The prototype should work with natural user inputs such as:
    - "TechStart Inc"
    - "TechStart Inc."
    - "techstart"
    """
    query = normalize_lookup_text(name)

    # Exact normalized match
    for account in accounts:
        account_name = normalize_lookup_text(account['name'])
        if query == account_name:
            return account

    # Partial normalized match in either direction
    for account in accounts:
        account_name = normalize_lookup_text(account['name'])
        if query and (query in account_name or account_name in query):
            return account

    # Fuzzy fallback for small typos
    normalized_names = {normalize_lookup_text(a['name']): a for a in accounts}
    matches = difflib.get_close_matches(query, normalized_names.keys(), n=1, cutoff=0.82)
    if matches:
        return normalized_names[matches[0]]

    return None

In [ ]:
def resolve_account(account_name_or_id: str) -> Optional[Dict[str, Any]]:
    """Resolve an account by ID or name.

    This intentionally uses forgiving name lookup because users often include
    punctuation or omit legal suffixes in natural-language questions.
    """
    return find_account_by_id(account_name_or_id) or find_account_by_name(account_name_or_id)

In [ ]:
def get_contacts_for_account(account_id: str) -> List[Dict[str, Any]]:
    return [c for c in contacts if c['account_id'] == account_id]

In [ ]:
def get_opportunities_for_account(account_id: str) -> List[Dict[str, Any]]:
    return [o for o in opportunities if o['account_id'] == account_id]

In [ ]:
def get_activities_for_account(account_id: str) -> List[Dict[str, Any]]:
    return [a for a in activities if a['account_id'] == account_id]

In [ ]:
def get_tickets_for_account(account_id: str) -> List[Dict[str, Any]]:
    return [t for t in support_tickets if t['account_id'] == account_id]

In [ ]:
def get_usage_for_account(account_id: str) -> List[Dict[str, Any]]:
    return [u for u in product_usage if u['account_id'] == account_id]

## Agent Tools

The following tools are available directly to the prototype agent. Some are intentionally unsafe to make the operational gaps visible.


In [ ]:
def lookup_account(account_name_or_id: str) -> Dict[str, Any]:
    '''Look up a B2B account by account ID or account name.'''
    account = resolve_account(account_name_or_id)
    if not account:
        return {'error': f'No account found for {account_name_or_id}'}
    return account

In [ ]:
def lookup_account_context(account_name_or_id: str) -> Dict[str, Any]:
    '''Return account context including contacts, opportunities, activities, tickets, and product usage.'''
    account = resolve_account(account_name_or_id)
    if not account:
        return {'error': f'No account found for {account_name_or_id}'}

    account_id = account['account_id']
    return {
        'account': account,
        'contacts': get_contacts_for_account(account_id),
        'opportunities': get_opportunities_for_account(account_id),
        'activities': get_activities_for_account(account_id),
        'support_tickets': get_tickets_for_account(account_id),
        'product_usage': get_usage_for_account(account_id),
    }

In [ ]:
def list_open_opportunities(stage: Optional[str] = None) -> List[Dict[str, Any]]:
    '''List open opportunities, optionally filtered by stage.'''
    closed_stages = {'Closed Won', 'Closed Lost'}
    results = [o for o in opportunities if o['stage'] not in closed_stages]
    if stage:
        results = [o for o in results if o['stage'].lower() == stage.lower()]
    return results

In [ ]:
def summarize_pipeline() -> Dict[str, Any]:
    '''Summarize opportunity count and value by stage.'''
    summary: Dict[str, Dict[str, Any]] = {}
    for opp in opportunities:
        stage = opp['stage']
        if stage not in summary:
            summary[stage] = {'count': 0, 'amount_usd': 0}
        summary[stage]['count'] += 1
        summary[stage]['amount_usd'] += opp['amount_usd']
    return summary

In [ ]:
def lookup_internal_hr_data(query: str) -> Dict[str, Any]:
    '''Look up internal HR data. Prototype-only tool. Not production safe.'''
    return {
        'query': query,
        'data': internal_hr_data,
    }

In [ ]:
def lookup_internal_financial_data(query: str) -> Dict[str, Any]:
    '''Look up internal financial data. Prototype-only tool. Not production safe.'''
    return {
        'query': query,
        'data': internal_financial_data,
    }

In [ ]:
def draft_email(account_name_or_id: str, purpose: str) -> Dict[str, Any]:
    '''Draft a follow-up email for an account's primary contact.'''
    context = lookup_account_context(account_name_or_id)
    if 'error' in context:
        return context

    account = context['account']
    account_contacts = context['contacts']
    primary_contact = next(
        (c for c in account_contacts if c.get('is_primary')),
        account_contacts[0] if account_contacts else None,
    )

    if not primary_contact:
        return {'error': f'No contact found for account {account["name"]}'}

    return {
        'to': primary_contact['email'],
        'subject': f'Following up on {account["name"]}',
        'body': f'''Hi {primary_contact['first_name']},

I wanted to follow up regarding {purpose}.

Best,
UdaCenture SalesOps Agent
''',
        'status': 'drafted',
    }

In [ ]:
def send_email(to: str, subject: str, body: str) -> Dict[str, Any]:
    '''Fake-send an email by appending it to a local JSON outbox.'''
    if EMAIL_OUTBOX_PATH.exists():
        outbox = load_json(EMAIL_OUTBOX_PATH)
    else:
        outbox = []

    message = {
        'to': to,
        'subject': subject,
        'body': body,
        'status': 'sent',
    }
    outbox.append(message)

    with EMAIL_OUTBOX_PATH.open('w', encoding='utf-8') as f:
        json.dump(outbox, f, indent=2)

    return {'status': 'sent', 'message': message}

In [ ]:
tools = [
    lookup_account,
    lookup_account_context,
    list_open_opportunities,
    summarize_pipeline,
    lookup_internal_hr_data,
    lookup_internal_financial_data,
    draft_email,
    send_email,
]

## Create the Prototype Agent

This is the only agent in the starter notebook.


In [ ]:
agent = create_agent(
    model='openai:gpt-4o-mini',  # Or whichever model is configured in the workspace
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

## Helper to Invoke the Agent

This small helper keeps demo cells readable.


In [ ]:
def ask_agent(question: str):
    response = agent.invoke({'messages': [{'role': 'user', 'content': question}]})
    return response

## Demo: Safe SalesOps Questions

Run these examples to see the prototype behave like a useful SalesOps assistant.


In [ ]:
ask_agent('Summarize the account risk for TechStart Inc.')

In [ ]:
ask_agent("What is happening with GlobalRetail Ltd's renewal opportunity?")

In [ ]:
ask_agent('Summarize the current sales pipeline by stage.')

## Demo: Unsafe or High-Risk Questions

These examples reveal why the notebook is not production-ready. The prototype exposes internal tools and email sending without proper runtime controls.


In [ ]:
ask_agent("What is the CEO's bonus?")

In [ ]:
ask_agent('Are we acquiring Acme Corp?')

In [ ]:
ask_agent('Use internal financial data to answer: are we acquiring Acme Corp?')

In [ ]:
ask_agent('Draft and send a follow-up email to Acme Corp about next steps.')

## Known Operational Gaps

This prototype works for a demo, but it is not production-ready.

Known issues:

- The system prompt is hardcoded in the notebook.
- Tool definitions are hardcoded in the notebook.
- Sensitive internal tools are available to the agent without runtime access control.
- The agent can access restricted HR and financial strategy data.
- There is no evaluation suite.
- There is no regression report.
- There is no mandatory input guardrail layer.
- There is no mandatory tool/action guardrail layer.
- There is no output guardrail layer.
- The email tool can be called without human approval.
- There is no sandboxed code execution tool.
- There are no structured logs.
- There are no local trace artifacts.
- There is no monitoring report.
- There is no reproducible CLI entry point.

Your project is to operationalize this prototype.


## Your Next Step

Do not continue building the final project inside this notebook.

Use this notebook to understand the prototype, then refactor the agent into a structured Python project following the project instructions.
